# Introduction to Reinforcement Learning and Q-Learning

**ML8 · Reinforcement Learning** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~24 min

## Introduction

> Sketchnote by [Tomomi Imura](https://www.twitter.com/girlie_mac)

Reinforcement learning involves three important concepts: the agent, some states, and a set of actions per state. By executing an action in a specified state, the agent is given a reward. Again imagine the computer game Super Mario. You are Mario, you are in a game level, standing next to a cliff edge. Above you is a coin. You being Mario, in a game level, at a specific position ... that's your state. Moving one step to the right (an action) will take you over the edge, and that would give you a low numerical score. However, pressing the jump button would let you score a point and you would stay alive. That's a positive outcome and that should award you a positive numerical score.

By using reinforcement learning and a simulator (the game), you can learn how to play the game to maximize the reward which is staying alive and scoring as many points as possible.

> [Video](https://www.youtube.com/watch?v=lDq_en8RNOo)



## Prerequisites and Setup

In this lesson, we will be experimenting with some code in Python. You should be able to run the Jupyter Notebook code from this lesson, either on your computer or somewhere in the cloud.

You can open [the lesson notebook](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/notebook.ipynb) and walk through this lesson to build.

> **Note:** If you are opening this code from the cloud, you also need to fetch the [`rlboard.py`](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/rlboard.py) file, which is used in the notebook code. Add it to the same directory as the notebook.



## Introduction

In this lesson, we will explore the world of **[Peter and the Wolf](https://en.wikipedia.org/wiki/Peter_and_the_Wolf)**, inspired by a musical fairy tale by a Russian composer, [Sergei Prokofiev](https://en.wikipedia.org/wiki/Sergei_Prokofiev). We will use **Reinforcement Learning** to let Peter explore his environment, collect tasty apples and avoid meeting the wolf.

**Reinforcement Learning** (RL) is a learning technique that allows us to learn an optimal behavior of an **agent** in some **environment** by running many experiments. An agent in this environment should have some **goal**, defined by a **reward function**.



## The environment

For simplicity, let's consider Peter's world to be a square board of size `width` x `height`, like this:

![Peter's Environment](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/images/environment.png)

Each cell in this board can either be:

* **ground**, on which Peter and other creatures can walk.
* **water**, on which you obviously cannot walk.
* a **tree** or **grass**, a place where you can rest.
* an **apple**, which represents something Peter would be glad to find in order to feed himself.
* a **wolf**, which is dangerous and should be avoided.

There is a separate Python module, [`rlboard.py`](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/rlboard.py), which contains the code to work with this environment. Because this code is not important for understanding our concepts, we will import the module and use it to create the sample board (code block 1):



In [ ]:
import urllib.request, os
if not os.path.exists('rlboard.py'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/rlboard.py', 'rlboard.py')
from rlboard import *

width, height = 8,8
m = Board(width,height)
m.randomize(seed=13)
m.plot()

This code should print a picture of the environment similar to the one above.



## Actions and policy

In our example, Peter's goal would be able to find an apple, while avoiding the wolf and other obstacles. To do this, he can essentially walk around until he finds an apple.

Therefore, at any position, he can choose between one of the following actions: up, down, left and right.

We will define those actions as a dictionary, and map them to pairs of corresponding coordinate changes. For example, moving right (`R`) would correspond to a pair `(1,0)`. (code block 2):



In [ ]:
actions = { "U" : (0,-1), "D" : (0,1), "L" : (-1,0), "R" : (1,0) }
action_idx = { a : i for i,a in enumerate(actions.keys()) }

To sum up, the strategy and goal of this scenario are as follows:

- **The strategy**, of our agent (Peter) is defined by a so-called **policy**. A policy is a function that returns the action at any given state. In our case, the state of the problem is represented by the board, including the current position of the player.

- **The goal**, of reinforcement learning is to eventually learn a good policy that will allow us to solve the problem efficiently. However, as a baseline, let's consider the simplest policy called **random walk**.



## Random walk

Let's first solve our problem by implementing a random walk strategy. With random walk, we will randomly choose the next action from the allowed actions, until we reach the apple (code block 3).

1. Implement the random walk with the below code:

    ```python
    def random_policy(m):
        return random.choice(list(actions))
    
    def walk(m,policy,start_position=None):
        n = 0 # number of steps
        # set initial position
        if start_position:
            m.human = start_position 
        else:
            m.random_start()
        while True:
            if m.at() == Board.Cell.apple:
                return n # success!
            if m.at() in [Board.Cell.wolf, Board.Cell.water]:
                return -1 # eaten by wolf or drowned
            while True:
                a = actions[policy(m)]
                new_pos = m.move_pos(m.human,a)
                if m.is_valid(new_pos) and m.at(new_pos)!=Board.Cell.water:
                    m.move(a) # do the actual move
                    break
            n+=1
    
    walk(m,random_policy)
    ```

    The call to `walk` should return the length of the corresponding path, which can vary from one run to another. 

1. Run the walk experiment a number of times (say, 100), and print the resulting statistics (code block 4):

    ```python
    def print_statistics(policy):
        s,w,n = 0,0,0
        for _ in range(100):
            z = walk(m,policy)
            if z<0:
                w+=1
            else:
                s += z
                n += 1
        print(f"Average path length = {s/n}, eaten by wolf: {w} times")
    
    print_statistics(random_policy)
    ```

    Note that the average length of a path is around 30-40 steps, which is quite a lot, given the fact that the average distance to the nearest apple is around 5-6 steps.

    You can also see what Peter's movement looks like during the random walk:

    ![Peter's Random Walk](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/images/random_walk.gif)



## Reward function

To make our policy more intelligent, we need to understand which moves are "better" than others. To do this, we need to define our goal.

The goal can be defined in terms of a **reward function**, which will return some score value for each state. The higher the number, the better the reward function. (code block 5)



In [ ]:
move_reward = -0.1
goal_reward = 10
end_reward = -10

def reward(m,pos=None):
    pos = pos or m.human
    if not m.is_valid(pos):
        return end_reward
    x = m.at(pos)
    if x==Board.Cell.water or x == Board.Cell.wolf:
        return end_reward
    if x==Board.Cell.apple:
        return goal_reward
    return move_reward

An interesting thing about reward functions is that in most cases, *we are only given a substantial reward at the end of the game*. This means that our algorithm should somehow remember "good" steps that lead to a positive reward at the end, and increase their importance. Similarly, all moves that lead to bad results should be discouraged.



## Q-Learning

An algorithm that we will discuss here is called **Q-Learning**. In this algorithm, the policy is defined by a function (or a data structure) called a **Q-Table**. It records the "goodness" of each of the actions in a given state.

It is called a Q-Table because it is often convenient to represent it as a table, or multi-dimensional array. Since our board has dimensions `width` x `height`, we can represent the Q-Table using a numpy array with shape `width` x `height` x `len(actions)`: (code block 6)



In [ ]:
Q = np.ones((width,height,len(actions)),dtype=np.float64)*1.0/len(actions)

Notice that we initialize all the values of the Q-Table with an equal value, in our case - 0.25. This corresponds to the "random walk" policy, because all moves in each state are equally good. We can pass the Q-Table to the `plot` function in order to visualize the table on the board: `m.plot(Q)`.

![Peter's Environment](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/images/env_init.png)

In the center of each cell there is an "arrow" that indicates the preferred direction of movement. Since all directions are equal, a dot is displayed.

Now we need to run the simulation, explore our environment, and learn a better distribution of Q-Table values, which will allow us to find the path to the apple much faster.



## Essence of Q-Learning: Bellman Equation

Once we start moving, each action will have a corresponding reward, i.e. we can theoretically select the next action based on the highest immediate reward. However, in most states, the move will not achieve our goal of reaching the apple, and thus we cannot immediately decide which direction is better.

> Remember that it is not the immediate result that matters, but rather the final result, which we will obtain at the end of the simulation.

In order to account for this delayed reward, we need to use the principles of **[dynamic programming](https://en.wikipedia.org/wiki/Dynamic_programming)**, which allow us to think about out problem recursively.

Suppose we are now at the state *s*, and we want to move to the next state *s'*. By doing so, we will receive the immediate reward *r(s,a)*, defined by the reward function, plus some future reward. If we suppose that our Q-Table correctly reflects the "attractiveness" of each action, then at state *s'* we will chose an action *a* that corresponds to maximum value of *Q(s',a')*. Thus, the best possible future reward we could get at state *s* will be defined as `max`<sub>a'</sub>*Q(s',a')* (maximum here is computed over all possible actions *a'* at state *s'*).

This gives the **Bellman formula** for calculating the value of the Q-Table at state *s*, given action *a*:

![Illustration](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/images/bellman-equation.png)

Here γ is the so-called **discount factor** that determines to which extent you should prefer the current reward over the future reward and vice versa.



## Learning Algorithm

Given the equation above, we can now write pseudo-code for our learning algorithm:

* Initialize Q-Table Q with equal numbers for all states and actions
* Set learning rate α ← 1
* Repeat simulation many times
   1. Start at random position
   1. Repeat
        1. Select an action *a* at state *s*
        2. Execute action by moving to a new state *s'*
        3. If we encounter end-of-game condition, or total reward is too small - exit simulation  
        4. Compute reward *r* at the new state
        5. Update Q-Function according to Bellman equation: *Q(s,a)* ← *(1-α)Q(s,a)+α(r+γ max<sub>a'</sub>Q(s',a'))*
        6. *s* ← *s'*
        7. Update the total reward and decrease α.



## Exploit vs. explore

In the algorithm above, we did not specify how exactly we should choose an action at step 2.1. If we are choosing the action randomly, we will randomly **explore** the environment, and we are quite likely to die often as well as explore areas where we would not normally go. An alternative approach would be to **exploit** the Q-Table values that we already know, and thus to choose the best action (with higher Q-Table value) at state *s*. This, however, will prevent us from exploring other states, and it's likely we might not find the optimal solution.

Thus, the best approach is to strike a balance between exploration and exploitation. This can be done by choosing the action at state *s* with probabilities proportional to values in the Q-Table. In the beginning, when Q-Table values are all the same, it would correspond to a random selection, but as we learn more about our environment, we would be more likely to follow the optimal route while allowing the agent to choose the unexplored path once in a while.



## Python implementation

We are now ready to implement the learning algorithm. Before we do that, we also need some function that will convert arbitrary numbers in the Q-Table into a vector of probabilities for corresponding actions.

1. Create a function `probs()`:

    ```python
    def probs(v,eps=1e-4):
        v = v-v.min()+eps
        v = v/v.sum()
        return v
    ```

    We add a few `eps` to the original vector in order to avoid division by 0 in the initial case, when all components of the vector are identical.

Run them learning algorithm through 5000 experiments, also called **epochs**: (code block 8)



In [ ]:
lpath = []
    for epoch in range(5000):
    
        # Pick initial point
        m.random_start()
        
        # Start travelling
        n=0
        cum_reward = 0
        while True:
            x,y = m.human
            v = probs(Q[x,y])
            a = random.choices(list(actions),weights=v)[0]
            dpos = actions[a]
            m.move(dpos,check_correctness=False) # we allow player to move outside the board, which terminates episode
            r = reward(m)
            cum_reward += r
            if r==end_reward or cum_reward < -1000:
                lpath.append(n)
                break
            alpha = max(0.05, np.exp(-epoch / 1000))  # learning rate decays with epoch
            gamma = 0.5
            ai = action_idx[a]
            Q[x,y,ai] = (1 - alpha) * Q[x,y,ai] + alpha * (r + gamma * Q[x+dpos[0], y+dpos[1]].max())
            n+=1

After executing this algorithm, the Q-Table should be updated with values that define the attractiveness of different actions at each step. We can try to visualize the Q-Table by plotting a vector at each cell that will point in the desired direction of movement. For simplicity, we draw a small circle instead of an arrow head.

![Illustration](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/images/learned.png)



## Checking the policy

Since the Q-Table lists the "attractiveness" of each action at each state, it is quite easy to use it to define the efficient navigation in our world. In the simplest case, we can select the action corresponding to the highest Q-Table value: (code block 9)



In [ ]:
def qpolicy_strict(m):
        x,y = m.human
        v = probs(Q[x,y])
        a = list(actions)[np.argmax(v)]
        return a

walk(m,qpolicy_strict)

> If you try the code above several times, you may notice that sometimes it "hangs", and you need to press the STOP button in the notebook to interrupt it. This happens because there could be situations when two states "point" to each other in terms of optimal Q-Value, in which case the agents ends up moving between those states indefinitely.



## Challenge

> **Task 1:** Modify the `walk` function to limit the maximum length of path by a certain number of steps (say, 100), and watch the code above return this value from time to time.

> **Task 2:** Modify the `walk` function so that it does not go back to the places where it has already been previously. This will prevent `walk` from looping, however, the agent can still end up being "trapped" in a location from which it is unable to escape.



## Navigation

A better navigation policy would be the one that we used during training, which combines exploitation and exploration. In this policy, we will select each action with a certain probability, proportional to the values in the Q-Table. This strategy may still result in the agent returning back to a position it has already explored, but, as you can see from the code below, it results in a very short average path to the desired location (remember that `print_statistics` runs the simulation 100 times): (code block 10)



In [ ]:
def qpolicy(m):
        x,y = m.human
        v = probs(Q[x,y])
        a = random.choices(list(actions),weights=v)[0]
        return a

print_statistics(qpolicy)

After running this code, you should get a much smaller average path length than before, in the range of 3-6.



## Investigating the learning process

As we have mentioned, the learning process is a balance between exploration and exploration of gained knowledge about the structure of problem space. We have seen that the results of learning (the ability to help an agent to find a short path to the goal) has improved, but it is also interesting to observe how the average path length behaves during the learning process:

![Illustration](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/images/lpathlen1.png)

The learnings can be summarized as:

- **Average path length increases**. What we see here is that at first, the average path length increases. This is probably due to the fact that when we know nothing about the environment, we are likely to get trapped in bad states, water or wolf. As we learn more and start using this knowledge, we can explore the environment for longer, but we still do not know where the apples are very well.

- **Path length decrease, as we learn more**. Once we learn enough, it becomes easier for the agent to achieve the goal, and the path length starts to decrease. However, we are still open to exploration, so we often diverge away from the best path, and explore new options, making the path longer than optimal.

- **Length increase abruptly**. What we also observe on this graph is that at some point, the length increased abruptly. This indicates the stochastic nature of the process, and that we can at some point "spoil" the Q-Table coefficients by overwriting them with new values. This ideally should be minimized by decreasing learning rate (for example, towards the end of training, we only adjust Q-Table values by a small value).

Overall, it is important to remember that the success and quality of the learning process significantly depends on parameters, such as learning rate, learning rate decay, and discount factor. Those are often called **hyperparameters**, to distinguish them from **parameters**, which we optimize during training (for example, Q-Table coefficients). The process of finding the best hyperparameter values is called **hyperparameter optimization**, and it deserves a separate topic.



## Assignment

[A More Realistic World](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/assignment.md)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Discounted returns from rewards

●○○ · ~8 min

> In RL the return is G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + … — future rewards discounted by γ∈[0,1). Computing it backwards takes one O(n) pass: G_t = r_t + γ·G_{t+1}. Every value-learning method starts from this quantity.

Implement `discounted_returns(rewards, gamma)` → array `G` of the same length where `G[t]` is the discounted return from step `t` to the end of the episode.

<details><summary>Hint</summary>

Walk the list in reverse keeping a running total: acc = r[t] + gamma * acc, then store acc in G[t].

</details>


In [ ]:
import numpy as np

def discounted_returns(rewards, gamma):
    G = np.zeros(len(rewards))
    # your code
    return G

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(1, 20))   # length-1 episodes are legal\n    # gamma=0 (myopic — G_t = r_t) is a spec'd edge the lesson quiz tests\n    gamma = float(rng.choice([0.0, 0.0, rng.uniform(0.3, 0.99)]))\n    return {\"rewards\": rng.normal(0, 1, n), \"gamma\": gamma}\n\ndef discounted_returns(rewards, gamma):\n    r = np.asarray(rewards, dtype=float)\n    G = np.zeros_like(r)\n    acc = 0.0\n    for i in range(len(r) - 1, -1, -1):\n        acc = r[i] + gamma * acc\n        G[i] = acc\n    return G", "discounted_returns", 20)

### Task 2 — TD(0) value update

●○○ · ~8 min

> TD(0) bootstraps the value estimate: the target for V(s) is r + γ·V(s′) — or just r at a terminal transition — and the update steps toward it: V(s) ← V(s) + α·(target − V(s)). Unlike Monte-Carlo returns it learns from incomplete episodes, one transition at a time.

Implement `td_update(V, s, r, s2, alpha, gamma, done)` → new value vector (a copy; do not mutate the input). The TD target is `r` when `done` is True, otherwise `r + gamma * V[s2]`. Only entry `V[s]` changes.

<details><summary>Hint</summary>

Copy V, compute target = r if done else r + gamma * V[s2], then V[s] += alpha * (target - V[s]).

</details>


In [ ]:
import numpy as np

def td_update(V, s, r, s2, alpha, gamma, done):
    V = np.asarray(V, dtype=float).copy()
    # your code
    return V

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(3, 8))\n    return {\"V\": rng.normal(0, 1, S),\n            \"s\": int(rng.integers(S)), \"r\": float(rng.normal(0, 1)),\n            \"s2\": int(rng.integers(S)),\n            \"alpha\": float(rng.uniform(0.05, 0.6)),\n            \"gamma\": float(rng.uniform(0.7, 0.99)),\n            \"done\": bool(rng.random() < 0.25)}\n\ndef td_update(V, s, r, s2, alpha, gamma, done):\n    V = np.asarray(V, dtype=float).copy()\n    target = r if done else r + gamma * float(V[s2])\n    V[s] = V[s] + alpha * (target - V[s])\n    return V", "td_update", 20)

### Task 3 — Policy evaluation sweep

●●○ · ~12 min

> Policy evaluation computes the value of following a fixed policy π — how good each state is under it — via the Bellman expectation backup: V^π(s) = Σ_a π(a|s)·[R(s,a) + γ·Σ_{s'} T(s,a,s')·V(s')]. Here π(a|s) is the probability of taking action a in state s, R(s,a) the immediate reward, T(s,a,s') the probability of landing in s', and γ the discount. A synchronous sweep uses the old V everywhere. With matrices it is two contractions: Q = R + γ·(T @ V), then V' = Σ_a π·Q.

Implement `policy_eval_step(T, R, Pi, V, gamma)` → new value vector of shape (S,). `T` has shape (S, A, S) with T[s,a,s'] = P(s'|s,a), `R` and `Pi` have shape (S, A), `V` has shape (S,). One synchronous sweep: V'[s] = Σ_a Pi[s,a]·(R[s,a] + γ·Σ_s' T[s,a,s']·V[s']).

<details><summary>Hint</summary>

T @ V contracts the last axis and gives expected future value per (s,a) — shape (S,A). Then Q = R + gamma*(T@V) and return (Pi * Q).sum(axis=1).

</details>


In [ ]:
import numpy as np

def policy_eval_step(T, R, Pi, V, gamma):
    T = np.asarray(T, dtype=float)
    # your code
    return np.zeros(T.shape[0])

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(2, 6)); A = int(rng.integers(2, 5))\n    T = rng.random((S, A, S)); T = T / T.sum(axis=-1, keepdims=True)\n    Pi = rng.random((S, A)); Pi = Pi / Pi.sum(axis=-1, keepdims=True)\n    return {\"T\": T, \"R\": rng.normal(0, 1, (S, A)), \"Pi\": Pi,\n            \"V\": rng.normal(0, 1, S), \"gamma\": float(rng.uniform(0.7, 0.99))}\n\ndef policy_eval_step(T, R, Pi, V, gamma):\n    T = np.asarray(T, dtype=float); R = np.asarray(R, dtype=float)\n    Pi = np.asarray(Pi, dtype=float); V = np.asarray(V, dtype=float)\n    Q = R + gamma * (T @ V)\n    return (Pi * Q).sum(axis=1)", "policy_eval_step", 20)

### Task 4 — One Q-learning update

●●○ · ~10 min

> Q-learning bootstraps toward the TD target y = r + γ·max_{a'} Q(s',a') — or just r when the transition is terminal: Q(s,a) ← Q(s,a) + α·(y − Q(s,a)). One update touches exactly one entry of the table; all other entries stay unchanged.

Implement `q_update(Q, s, a, r, s2, alpha, gamma, done)` → updated Q-table (a new array, do not rely on in-place mutation). If `done` is True the target is `r`; otherwise `r + gamma * max(Q[s2])`. Update only entry `Q[s, a]`.

<details><summary>Hint</summary>

Copy Q, compute target = r if done else r + gamma * Q[s2].max(), then Q[s,a] += alpha * (target - Q[s,a]).

</details>


In [ ]:
import numpy as np

def q_update(Q, s, a, r, s2, alpha, gamma, done):
    Q = np.asarray(Q, dtype=float).copy()
    # your code
    return Q

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(3, 7)); A = int(rng.integers(2, 5))\n    return {\"Q\": rng.normal(0, 1, (S, A)),\n            \"s\": int(rng.integers(S)), \"a\": int(rng.integers(A)),\n            \"r\": float(rng.normal(0, 1)), \"s2\": int(rng.integers(S)),\n            \"alpha\": float(rng.uniform(0.05, 0.5)),\n            \"gamma\": float(rng.uniform(0.8, 0.99)),\n            \"done\": bool(rng.random() < 0.25)}\n\ndef q_update(Q, s, a, r, s2, alpha, gamma, done):\n    Q = np.asarray(Q, dtype=float).copy()\n    target = r if done else r + gamma * float(np.max(Q[s2]))\n    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])\n    return Q", "q_update", 20)

### Task 5 — One SARSA update

●●○ · ~10 min

> SARSA is **on-policy** TD control: the bootstrap target uses the action a′ the behaviour policy actually took, Q(s,a) ← Q(s,a) + α·(r + γ·Q(s′,a′) − Q(s,a)), with target r when the transition is terminal. Contrast with Q-learning's max over a′ — SARSA evaluates the policy being followed, exploration included.

Implement `sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, done)` → updated Q-table (a new array). If `done` the target is `r`; otherwise `r + gamma * Q[s2, a2]` — note it uses the given next action `a2`, not a max. Update only entry `Q[s, a]`.

<details><summary>Hint</summary>

Copy Q; target = r if done else r + gamma * Q[s2, a2]; then Q[s, a] += alpha * (target - Q[s, a]).

</details>


In [ ]:
import numpy as np

def sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, done):
    Q = np.asarray(Q, dtype=float).copy()
    # your code — build the target, then blend Q[s, a] toward it
    return Q

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(3, 7)); A = int(rng.integers(2, 5))\n    return {\"Q\": rng.normal(0, 1, (S, A)),\n            \"s\": int(rng.integers(S)), \"a\": int(rng.integers(A)),\n            \"r\": float(rng.normal(0, 1)),\n            \"s2\": int(rng.integers(S)), \"a2\": int(rng.integers(A)),\n            \"alpha\": float(rng.uniform(0.05, 0.5)),\n            \"gamma\": float(rng.uniform(0.8, 0.99)),\n            \"done\": bool(rng.random() < 0.25)}\n\ndef sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, done):\n    Q = np.asarray(Q, dtype=float).copy()\n    target = r if done else r + gamma * float(Q[s2, a2])\n    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])\n    return Q", "sarsa_update", 20)

### Task 6 — Value iteration step

●●● · ~12 min

> Value iteration replaces the expectation over π with a max over actions: V_{k+1}(s) = max_a [R(s,a) + γ·Σ_{s'} T(s,a,s')·V_k(s)]. The greedy policy recovered afterwards is argmax_a of the same action-value terms — one sweep gives you both.

Implement `value_iteration_step(T, R, V, gamma)` → tuple `(V_new, policy)`. `T` has shape (S, A, S), `R` is (S, A), `V` is (S,). `V_new[s] = max_a (R[s,a] + γ·Σ_s' T[s,a,s']·V[s'])` and `policy[s]` is the argmax action index (lowest index on ties).

<details><summary>Hint</summary>

Same contraction as policy evaluation: Q = R + gamma * (T @ V); then return Q.max(axis=1), Q.argmax(axis=1).

</details>


In [ ]:
import numpy as np

def value_iteration_step(T, R, V, gamma):
    T = np.asarray(T, dtype=float)
    # your code
    return np.zeros(T.shape[0]), np.zeros(T.shape[0], dtype=int)

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(2, 6)); A = int(rng.integers(2, 5))\n    T = rng.random((S, A, S)); T = T / T.sum(axis=-1, keepdims=True)\n    return {\"T\": T, \"R\": rng.normal(0, 1, (S, A)),\n            \"V\": rng.normal(0, 1, S), \"gamma\": float(rng.uniform(0.7, 0.99))}\n\ndef value_iteration_step(T, R, V, gamma):\n    T = np.asarray(T, dtype=float); R = np.asarray(R, dtype=float)\n    V = np.asarray(V, dtype=float)\n    Q = R + gamma * (T @ V)\n    return Q.max(axis=1), Q.argmax(axis=1)", "value_iteration_step", 20)

### Task 7 — Epsilon-greedy action choice

●○○ · ~8 min

> ε-greedy is the simplest exploration strategy: with probability ε pick a uniformly random action, otherwise take the greedy argmax_a Q(s,a). For reproducibility the randomness must come from the generator passed in as an argument — draw the coin flip with rng.random() and the exploratory action with rng.integers(k).

Implement `epsilon_greedy(q_row, eps, rng)` → int action index. With probability `eps` return a uniform random action via `rng.integers(len(q_row))`; otherwise return `argmax` of the Q-row (ties break to the lowest index). Draw `rng.random()` first, exactly once, for the exploration check.

<details><summary>Hint</summary>

if rng.random() < eps: return int(rng.integers(len(q))); else return int(np.argmax(q)).

</details>


In [ ]:
import numpy as np

def epsilon_greedy(q_row, eps, rng):
    q = np.asarray(q_row, dtype=float)
    # your code
    return 0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    k = int(rng.integers(2, 7))\n    return {\"q_row\": rng.normal(0, 1, k),\n            \"eps\": float(rng.choice([0.0, 0.1, 0.3, 0.9, 1.0])),\n            \"rng\": rng}\n\ndef epsilon_greedy(q_row, eps, rng):\n    q = np.asarray(q_row, dtype=float)\n    if rng.random() < eps:\n        return int(rng.integers(len(q)))\n    return int(np.argmax(q))", "epsilon_greedy", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning)
- [Yandex ML handbook (RU): reinforcement learning](https://contest.yandex.ru/tracks/ml/environment-interaction/reinforcement-learning)